# HỆ THỐNG AI DỰ ĐOÁN ĐIỂM SỐ & GỢI Ý PHIM CÁ NHÂN HÓA (TMDB)

Notebook này trình bày chi tiết quy trình xây dựng một hệ thống **Machine Learning** hoàn chỉnh từ bước tiền xử lý dữ liệu đến huấn luyện, đánh giá và triển khai ứng dụng tương tác.

### Kiến trúc 3 mô hình & thuật toán cốt lõi trong hệ thống:
1. **Mô hình Dự đoán điểm đánh giá phim (`LinearRegression`):**
   - Học mối quan hệ tuyến tính giữa **Năm phát hành**, **Thời lượng** và **15 Thể loại phổ biến nhất** để dự đoán điểm `vote_average` (thang điểm $0 - 10$).
2. **Thuật toán Gợi ý phim theo nội dung (Content-Based Filtering với Exponential Decay):**
   - Đo lường độ tương đồng giữa tiêu chí người dùng nhập và toàn bộ kho phim thông qua **Độ phủ tập hợp thể loại** và **Hàm suy giảm mũ (Exponential Decay)** để xếp hạng **Top 10 phim hợp sở thích nhất**.
3. **Mô hình Học khẩu vị tương tác (`Ridge Regression` - L2 Regularization) & Giải thích AI (Explainable AI):**
   - Học trực tiếp từ phản hồi **Thích (`+1`) / Không thích (`-1`)** của người dùng trên 10 phim đầu tiên để tìm ra **Vector khẩu vị cá nhân $w$**, từ đó gợi ý thêm **Top 5 phim mới** kèm lời giải thích cụ thể vì sao AI chọn từng phim đó.


## Khai báo thư viện & Cấu hình hằng số

* **`pandas` & `numpy`**: Xử lý dữ liệu dạng bảng (DataFrame) và tính toán đại số tuyến tính (ma trận, vector) tốc độ cao.
* **`train_test_split`**: Chia tách bộ dữ liệu ngẫu nhiên thành các tập Train / Validation / Test.
* **`StandardScaler`**: Chuẩn hóa đặc trưng về phân phối chuẩn ($\mu = 0, \sigma = 1$).
* **`LinearRegression`**: Thuật toán Hồi quy tuyến tính (Bình phương tối tiểu thông thường - OLS).
* **`Ridge`**: Thuật toán Hồi quy tuyến tính có điều chuẩn L2 (Ridge Regression), chuyên trị bài toán học từ số lượng mẫu ít (Few-shot learning) mà không bị học vẹt (Overfitting).
* **`mean_absolute_error`, `mean_squared_error`, `r2_score`**: Bộ 3 thước đo tiêu chuẩn đánh giá mô hình hồi quy.

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CSV = 'TMDB_movie_dataset_cleaned.csv'
MIN_VOTES = 20      # Chỉ giữ phim có ít nhất 20 lượt vote
N_GENRES = 15       # Số thể loại phổ biến nhất đưa vào mô hình


## BƯỚC 1: Đọc dữ liệu, Lọc nhiễu (Data Cleaning) & Mã hóa đặc trưng (Feature Engineering)

### 1.1. Nguyên lý lọc nhiễu dữ liệu
* **`vote_count >= 20`**: Trong thống kê, những bộ phim có quá ít lượt bình chọn (ví dụ 1-5 lượt vote) có phương sai điểm số cực lớn (dễ bị 10.0 hoặc 1.0 tuyệt đối). Lọc từ 20 lượt vote trở lên giúp nhãn mục tiêu `vote_average` hội tụ về giá trị thực, giảm nhiễu cho mô hình học máy.
* **`vote_average > 0`**: Loại bỏ các bộ phim chưa được chấm điểm.
* **`runtime.between(1, 400)` & `dropna()`**: Loại bỏ dữ liệu rác (phim có thời lượng 0 phút hoặc dài bất thường trên 400 phút) và các bản ghi bị thiếu năm phát hành hoặc thời lượng.

### 1.2. Thuật toán Multi-label One-Hot Encoding (Mã hóa nhị phân đa nhãn)
Cột `genres` lưu chuỗi nhiều thể loại cách nhau bởi dấu phẩy (VD: `"Action, Science Fiction, Drama"`). Để chuyển thành số học:
1. Tách chuỗi của từng phim thành tập hợp `set` (`gset`) để loại bỏ khoảng trắng thừa và phần tử trùng lặp.
2. Thống kê tần suất xuất hiện của mọi thể loại và lấy ra **15 thể loại phổ biến nhất** (`top_genres`).
3. Tạo 15 cột nhị phân `G_<Tên_Thể_Loại>`: nhận giá trị `1` nếu phim thuộc thể loại đó và `0` nếu không thuộc. Khác với One-Hot thông thường (chỉ có một số 1 trên mỗi dòng), ở đây một bộ phim có thể có nhiều cột thể loại cùng bằng `1`.

In [10]:
# =====================================================================
# BƯỚC 1: ĐỌC VÀ LỌC DỮ LIỆU
# =====================================================================
cac_cot = ['title', 'vote_average', 'vote_count', 'genres', 'release_year', 'runtime']
df = pd.read_csv(CSV, usecols=cac_cot)

# Lọc dữ liệu theo từng tiêu chí rõ ràng
df = df[df['vote_count'] >= MIN_VOTES]              # Phim có từ 20 lượt vote trở lên
df = df[df['vote_average'] > 0]                     # Điểm trung bình > 0
df = df[df['runtime'].between(1, 400)]              # Thời lượng từ 1 đến 400 phút
df = df.dropna(subset=['release_year', 'runtime'])  # Bỏ phim bị thiếu năm hoặc thời lượng

print(f"Số phim dùng để huấn luyện: {len(df):,}")

# Tách chuỗi thể loại của từng phim (VD: "Action, Drama" -> {"Action", "Drama"})
danh_sach_gset = []
tat_ca_the_loai = []

for chuoi in df['genres'].fillna(''):
    tap_the_loai = set()
    for tu in chuoi.split(','):
        ten_tl = tu.strip()
        if ten_tl != '':
            tap_the_loai.add(ten_tl)

    danh_sach_gset.append(tap_the_loai)
    tat_ca_the_loai.extend(tap_the_loai)

df['gset'] = danh_sach_gset

# Tìm 15 thể loại xuất hiện nhiều nhất
top_genres = pd.Series(tat_ca_the_loai).value_counts().head(N_GENRES).index.tolist()

# Tạo các cột 0/1 cho 15 thể loại phổ biến (có thể loại đó = 1, không có = 0)
for g in top_genres:
    cot_0_1 = []
    for tap_the_loai in df['gset']:
        if g in tap_the_loai:
            cot_0_1.append(1)
        else:
            cot_0_1.append(0)
    df['G_' + g] = cot_0_1

Số phim dùng để huấn luyện: 49,768



## BƯỚC 2: Chia tập dữ liệu Train - Validation - Test (70% - 15% - 15%)

* **Ma trận đặc trưng $X$ (17 cột):** `release_year`, `runtime` và 15 cột `G_<Thể_loại>`.
* **Vector mục tiêu $y$ (1 cột):** `vote_average`.
* **Vai trò của 3 tập dữ liệu:**
  * **Tập Train (70%):** Dùng để mô hình học các tham số (trọng số hồi quy và thống kê chuẩn hóa).
  * **Tập Validation (15%):** Tập trung gian phục vụ việc tinh chỉnh siêu tham số (Hyperparameter Tuning) hoặc kiểm tra sớm nếu cần mở rộng mô hình.
  * **Tập Test (15%):** Giữ kín hoàn toàn trong suốt quá trình học, chỉ dùng ở Bước 5 để đo lường khả năng tổng quát hóa thực tế của AI.
* **Toán học chia tỷ lệ:**
  * Lần cắt 1: Giữ lại `15%` làm `X_test`, phần tạm `X_temp` còn `85%`.
  * Lần cắt 2: Cắt `15/85` của `X_temp` làm `X_val` (tức $85\% \times \frac{15}{85} = 15\%$ tổng dữ liệu), phần còn lại là `X_train` ($85\% \times \frac{70}{85} = 70\%$ tổng dữ liệu).

In [11]:
# =====================================================================
# BƯỚC 2: CHIA TẬP TRAIN - VAL - TEST (70% - 15% - 15%)
# =====================================================================
features = ['release_year', 'runtime']
for g in top_genres:
    features.append('G_' + g)

X = df[features]
y = df['vote_average']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(15/85), random_state=42)


## BƯỚC 3 & 4: Chuẩn hóa dữ liệu (`StandardScaler`) & Huấn luyện mô hình (`LinearRegression`)

### 3.1. Thuật toán Chuẩn hóa Z-score (`StandardScaler`)
Các đặc trưng ban đầu không cùng đơn vị đo: `release_year` có giá trị hàng nghìn ($\approx 2000$), `runtime` hàng trăm ($\approx 100$), còn thể loại chỉ là $0$ hoặc $1$.
`StandardScaler` biến đổi từng cột đặc trưng $j$ theo công thức:
$$z_{i,j} = \frac{x_{i,j} - \mu_j}{\sigma_j}$$
Trong đó $\mu_j$ là trung bình cộng và $\sigma_j$ là độ lệch chuẩn của cột $j$ **chỉ tính trên tập Train**.
* **Tại sao giúp ích cho mô hình?** Sau khi chuẩn hóa, mọi đặc trưng đều có trung bình bằng $0$ và độ lệch chuẩn bằng $1$. Nhờ vậy:
  1. Các hệ số hồi quy `model.coef_` ở Bước 5 phản ánh công bằng mức độ quan trọng của từng yếu tố.
  2. Phép tính khoảng cách và điều chuẩn L2 (`Ridge`) ở Bước 7 & 8 không bị thao túng bởi cột có con số lớn như năm phát hành.
* **Nguyên tắc chống rò rỉ dữ liệu (Preventing Data Leakage):** Chỉ gọi `fit_transform()` trên `X_train`, và chỉ gọi `transform()` trên `X_val`, `X_test`.

### 3.2. Mô hình Hồi quy tuyến tính (`LinearRegression`)
Mô hình giả định điểm số dự đoán $\hat{y}$ là tổ hợp tuyến tính của 17 đặc trưng đã chuẩn hóa:
$$\hat{y} = b_0 + b_1 z_1 + b_2 z_2 + \dots + b_{17} z_{17}$$
* **$b_0$ (`intercept_`):** Hệ số chặn (điểm trung bình gốc khi mọi đặc trưng $z_j = 0$).
* **$b_1, \dots, b_{17}$ (`coef_`):** Hệ số góc của từng đặc trưng.
* **Thuật toán tối ưu OLS (Ordinary Least Squares):** Tìm vector hệ số $b$ sao cho tổng bình phương phần dư (Residual Sum of Squares - RSS) đạt cực tiểu:
  $$\min_{b_0, b} \sum_{i=1}^{N_{\text{train}}} \left(y_i - \left(b_0 + \sum_{j=1}^{17} b_j z_{i,j}\right)\right)^2$$

In [12]:
# =====================================================================
# BƯỚC 3 & 4: CHUẨN HÓA DỮ LIỆU VÀ HUẤN LUYỆN MÔ HÌNH
# =====================================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](17,)","[-0.08, 0.23, 0.13,..., 0.04, 0.19, 0.02]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,6.259
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,17
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(17)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](17,)","[277.69,255.74,229.59,...,139.19,122.55,107.71]"



## BƯỚC 5: Đánh giá hiệu năng mô hình (Model Evaluation)

Hàm `np.clip(model.predict(X_test_scaled), 0, 10)` giới hạn đầu ra trong đoạn $[0, 10]$ đúng với thang điểm của TMDB.

### Ý nghĩa toán học của các chỉ số đánh giá:
1. **MAE (Mean Absolute Error - Sai số tuyệt đối trung bình):**
   $$\text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|$$
   Biểu thị trung bình dự đoán của AI lệch bao nhiêu điểm so với điểm thực tế.
2. **RMSE (Root Mean Squared Error - Căn bậc hai sai số bình phương trung bình):**
   $$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}$$
   Do bình phương sai số trước khi lấy trung bình, RMSE phạt rất nặng các trường hợp đoán sai lệch lớn (Outliers).
3. **So sánh với Baseline (Mô hình ngây thơ):**
   Baseline là mô hình không học gì cả mà luôn đoán mọi bộ phim có điểm bằng đúng trung bình cộng của tập Train ($\bar{y}_{\text{train}}$). Tỷ lệ giảm sai số:
   $$\text{Giảm sai số (\%)} = \left(1 - \frac{\text{RMSE}_{\text{AI}}}{\text{RMSE}_{\text{Baseline}}}\right) \times 100\%$$
4. **$R^2$ Score (Coefficient of Determination - Hệ số xác định):**
   $$R^2 = 1 - \frac{\sum_{i=1}^{N} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{N} (y_i - \bar{y}_{\text{test}})^2}$$
   Cho biết tỷ lệ phần trăm sự biến thiên của điểm đánh giá phim được giải thích bởi 17 đặc trưng đầu vào ($R^2 = 1$ là hoàn hảo, $R^2 = 0$ tương đương đoán trung bình).
5. **Đánh giá theo phân khúc (Subgroup Analysis) & Giải thích hệ số (`coef_`):**
   - Kiểm tra độ ổn định của mô hình qua từng **Thập niên** (`1920s -> 2020s`) và **Nhóm thời lượng** (`<90 phút`, `90-120 phút`, `120-150 phút`, `>150 phút`) với ngưỡng tối thiểu 30 phim/nhóm để đảm bảo ý nghĩa thống kê.
   - In 10 hệ số có trị tuyệt đối $|b_j|$ lớn nhất để biết yếu tố nào làm tăng điểm phim mạnh nhất ($b_j > 0$) và yếu tố nào kéo điểm phim xuống ($b_j < 0$).

In [13]:
# =====================================================================
# BƯỚC 5: ĐÁNH GIÁ MÔ HÌNH
# =====================================================================
y_test_pred = np.clip(model.predict(X_test_scaled), 0, 10)
print("\n             ĐÁNH GIÁ HIỆU NĂNG MÔ HÌNH (MODEL METRICS)           ")

# 1. Tổng thể (so với baseline: luôn đoán điểm trung bình của tập train)
mae_all = mean_absolute_error(y_test, y_test_pred)
rmse_all = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_all = r2_score(y_test, y_test_pred)

diem_tb_train = y_train.mean()
du_doan_baseline = np.full(len(y_test), diem_tb_train)
rmse_base = np.sqrt(mean_squared_error(y_test, du_doan_baseline))
giam_sai_so = (1 - rmse_all / rmse_base) * 100

print(f"[TỔNG THỂ] MAE: {mae_all:.2f} điểm | RMSE: {rmse_all:.2f} | R² Score: {r2_all:.4f}")
print(f"[BASELINE] luôn đoán {diem_tb_train:.2f}: RMSE = {rmse_base:.2f}  ->  mô hình giảm sai số {giam_sai_so:.1f}%")

df_eval = df.loc[X_test.index, ['title', 'release_year', 'runtime']].copy()
df_eval['Điểm_Thực_Tế'] = y_test
df_eval['Điểm_AI_Đoán'] = y_test_pred

# 2. Theo thập niên
print("\n[THEO THẬP NIÊN]")
df_eval['Thập_niên'] = (df_eval['release_year'] // 10 * 10).astype(int)
for dec in sorted(df_eval['Thập_niên'].unique()):
    sub_df = df_eval[df_eval['Thập_niên'] == dec]
    if len(sub_df) >= 30:
        mae_d = mean_absolute_error(sub_df['Điểm_Thực_Tế'], sub_df['Điểm_AI_Đoán'])
        r2_d = r2_score(sub_df['Điểm_Thực_Tế'], sub_df['Điểm_AI_Đoán'])
        print(f"  - Thập niên {dec}s ({len(sub_df)} phim): MAE = {mae_d:.2f} | R² = {r2_d:.4f}")

# 3. Theo phân khúc thời lượng
print("\n[THEO PHÂN KHÚC THỜI LƯỢNG]")
labels = ['<90 phút', '90-120 phút', '120-150 phút', '>150 phút']
df_eval['Nhóm_TL'] = pd.cut(df_eval['runtime'], bins=[0, 90, 120, 150, float('inf')], labels=labels)
for group in labels:
    sub_df = df_eval[df_eval['Nhóm_TL'] == group]
    if len(sub_df) >= 30:
        mae_g = mean_absolute_error(sub_df['Điểm_Thực_Tế'], sub_df['Điểm_AI_Đoán'])
        r2_g = r2_score(sub_df['Điểm_Thực_Tế'], sub_df['Điểm_AI_Đoán'])
        print(f"  - {group} ({len(sub_df)} phim): MAE = {mae_g:.2f} | R² = {r2_g:.4f}")

# 4. Hệ số của mô hình
print("\n[HỆ SỐ MÔ HÌNH]  Điểm = b0 + b1*x1 + b2*x2 + ...")
coef = pd.Series(model.coef_, index=features).sort_values(key=abs, ascending=False)
print(f"  b0 (điểm trung bình) = {model.intercept_:.3f}")
for name, c in coef.head(10).items():
    print(f"  {name:<22} {c:+.3f}")


             ĐÁNH GIÁ HIỆU NĂNG MÔ HÌNH (MODEL METRICS)           
[TỔNG THỂ] MAE: 0.65 điểm | RMSE: 0.83 | R² Score: 0.2206
[BASELINE] luôn đoán 6.26: RMSE = 0.94  ->  mô hình giảm sai số 11.7%

[THEO THẬP NIÊN]
  - Thập niên 1920s (46 phim): MAE = 0.77 | R² = -0.2374
  - Thập niên 1930s (113 phim): MAE = 0.57 | R² = -0.0530
  - Thập niên 1940s (178 phim): MAE = 0.48 | R² = -0.0354
  - Thập niên 1950s (262 phim): MAE = 0.60 | R² = 0.2102
  - Thập niên 1960s (332 phim): MAE = 0.57 | R² = 0.1659
  - Thập niên 1970s (409 phim): MAE = 0.64 | R² = 0.2267
  - Thập niên 1980s (543 phim): MAE = 0.65 | R² = 0.2231
  - Thập niên 1990s (665 phim): MAE = 0.60 | R² = 0.2285
  - Thập niên 2000s (1414 phim): MAE = 0.67 | R² = 0.2137
  - Thập niên 2010s (2661 phim): MAE = 0.66 | R² = 0.2266
  - Thập niên 2020s (762 phim): MAE = 0.67 | R² = 0.0909

[THEO PHÂN KHÚC THỜI LƯỢNG]
  - <90 phút (2643 phim): MAE = 0.69 | R² = 0.2175
  - 90-120 phút (3877 phim): MAE = 0.61 | R² = 0.1822
  - 120-150 phút (720


## BƯỚC 6: Xuất kết quả dự đoán và bộ hệ số ra file CSV

* Tính cột `Sai_Số = Điểm_AI_Đoán - Điểm_Thực_Tế` (làm tròn 3 chữ số thập phân) để phục vụ việc vẽ biểu đồ phân phối lỗi hoặc kiểm tra các bộ phim có chênh lệch lớn nhất.
* Sử dụng `encoding='utf-8-sig'` (UTF-8 kèm BOM) giúp hiển thị tiếng Việt có dấu chuẩn xác khi mở trực tiếp bằng Microsoft Excel.

In [14]:
# =====================================================================
# BƯỚC 6: XUẤT KẾT QUẢ DỰ ĐOÁN CHI TIẾT RA FILE CSV
# =====================================================================
df_eval['Sai_Số'] = (df_eval['Điểm_AI_Đoán'] - df_eval['Điểm_Thực_Tế']).round(3)
df_eval.drop(columns=['Nhóm_TL']).to_csv('ket_qua_du_doan.csv', index=False, encoding='utf-8-sig')
coef.rename('he_so').to_csv('he_so_mo_hinh.csv', encoding='utf-8-sig')
print("\nĐã lưu ket_qua_du_doan.csv và he_so_mo_hinh.csv")


Đã lưu ket_qua_du_doan.csv và he_so_mo_hinh.csv



## BƯỚC 7: Chuẩn bị "Kho phim" & Xây dựng các hàm toán học hỗ trợ Gợi ý

Để vòng lặp tương tác ở Bước 8 phản hồi tức thì (Real-time), toàn bộ dữ liệu kho phim được chuẩn hóa sẵn vào ma trận `X_all_scaled` và trích xuất ra các mảng NumPy (`tieu_de`, `nam_tat_ca`, `tl_tat_ca`, `diem_that`, `diem_ai`).

### Giải thích chi tiết các hàm hỗ trợ:
1. **`do_khop(gia_tri, muc_tieu, thang_do)` — Hàm suy giảm mũ (Exponential Decay Similarity):**
   $$S(x, x_0) = e^{-\frac{|x - x_0|}{\tau}}$$
   - Đo độ gần giữa giá trị của phim ($x$) và mong muốn của người dùng ($x_0$) với thang đo $\tau$ (`thang_do`).
   - Khi $x = x_0$: khoảng cách bằng $0 \Rightarrow S = e^0 = 1.0$ (khớp hoàn toàn).
   - Khi $|x - x_0| = \tau$ (ví dụ lệch 10 năm hoặc lệch 30 phút): điểm giảm còn $e^{-1} \approx 0.368$.
   - Ưu điểm vượt trội so với khoảng cách tuyến tính: hàm mũ luôn dương trong khoảng $(0, 1]$, giảm nhanh ở vùng lân cận mục tiêu và thoải dần ở xa, không bao giờ sinh ra điểm âm.
2. **`chuan_hoa_z(mang_diem)` — Chuẩn hóa Z-Score cho mảng điểm:**
   $$Z(v) = \frac{v - \mu_v}{\sigma_v + 10^{-9}}$$
   Đưa hai đại lượng khác bản chất (`diem_khau_vi` và `diem_that`) về cùng một mặt bằng đơn vị độ lệch chuẩn trước khi kết hợp tuyến tính. Hằng số $10^{-9}$ (`1e-9`) ngăn chặn lỗi chia cho 0 khi mọi phần tử trong mảng bằng nhau.
3. **`ten_dep(ten_cot, vi_tri_phim)` — Hàm ánh xạ giải thích (Feature-to-Text Mapping):**
   Chuyển tên cột kỹ thuật (`G_Action`, `release_year`, `runtime`) thành cụm từ tiếng Việt kèm theo đúng số năm phát hành và số phút thực tế của bộ phim tại vị trí `vi_tri_phim`.
4. **`doc_so_phim(chuoi_nhap, toi_da)`:**
   Chuẩn hóa chuỗi người dùng nhập (chấp nhận cả dấu phẩy `"1, 3, 5"` lẫn khoảng trắng `"1 3 5"`), kiểm tra tính hợp lệ (`1 <= số <= 10`), loại bỏ số trùng lặp và đổi từ số thứ tự hiển thị (`1-10`) sang chỉ số mảng Python (`0-9`).
5. **`in_danh_sach(danh_sach_vi_tri, ghi_chu=None)`:**
   Hàm hiển thị danh sách phim (Tên phim, năm, thời lượng, thể loại, điểm thật, điểm AI đoán và lời giải thích nếu có) gọn gàng ra màn hình.

In [15]:
# =====================================================================
# BƯỚC 7: CHUẨN BỊ "KHO PHIM" ĐỂ DỰ ĐOÁN VÀ GỢI Ý
# =====================================================================
X_all_scaled = scaler.transform(df[features])
df['AI_Đoán'] = np.clip(model.predict(X_all_scaled), 0, 10)

# Lấy sẵn thông tin toàn bộ phim ra mảng để tra cứu nhanh
tieu_de = df['title'].fillna('').astype(str).values
nam_tat_ca = df['release_year'].values.astype(float)
tl_tat_ca = df['runtime'].values.astype(float)
diem_that = df['vote_average'].values
diem_ai = df['AI_Đoán'].values
the_loai_phim_goc = df['genres'].fillna('').values

# Đổi thể loại của từng phim sang chữ thường để dễ so khớp
the_loai_tung_phim = []
for tap_tl in df['gset']:
    tap_chu_thuong = set()
    for g in tap_tl:
        tap_chu_thuong.add(g.lower())
    the_loai_tung_phim.append(tap_chu_thuong)


# --- CÁC HÀM HỖ TRỢ ĐƠN GIẢN ---
def do_khop(gia_tri, muc_tieu, thang_do):
    """Tính độ gần giữa giá trị của phim và mong muốn của người dùng (từ 0 đến 1)."""
    khoang_cach = np.abs(gia_tri - muc_tieu)
    return np.exp(-khoang_cach / thang_do)


def chuan_hoa_z(mang_diem):
    """Đưa mảng điểm về cùng một thang đo chuẩn để cộng gộp công bằng."""
    return (mang_diem - mang_diem.mean()) / (mang_diem.std() + 1e-9)


def ten_dep(ten_cot, vi_tri_phim):
    """Đổi tên cột kỹ thuật sang câu tiếng Việt có kèm số năm / số phút."""
    if ten_cot.startswith('G_'):
        return 'thể loại ' + ten_cot[2:]
    if ten_cot == 'release_year':
        return f'năm phát hành {int(nam_tat_ca[vi_tri_phim])}'
    if ten_cot == 'runtime':
        return f'thời lượng {int(tl_tat_ca[vi_tri_phim])} phút'
    return ten_cot


def doc_so_phim(chuoi_nhap, toi_da):
    """Đổi chuỗi '1 3 5' người dùng nhập thành danh sách vị trí [0, 2, 4]."""
    ket_qua = []
    chuoi_sach = chuoi_nhap.replace(',', ' ')
    for tu in chuoi_sach.split():
        if tu.isdigit():
            so_thu_tu = int(tu)
            vi_tri = so_thu_tu - 1
            if 1 <= so_thu_tu <= toi_da and vi_tri not in ket_qua:
                ket_qua.append(vi_tri)
    return ket_qua


def in_danh_sach(danh_sach_vi_tri, ghi_chu=None):
    """In danh sách phim ra màn hình một cách gọn gàng."""
    for stt, i in enumerate(danh_sach_vi_tri, 1):
        ten_phim = tieu_de[i][:50]
        nam_phim = int(nam_tat_ca[i])
        phut_phim = int(tl_tat_ca[i])
        the_loai = the_loai_phim_goc[i] or '?'

        print(f"{stt:>2}. {ten_phim} ({nam_phim}, {phut_phim} phút)")
        print(f"      Thể loại: {the_loai}")
        print(f"      Điểm thật {diem_that[i]:.1f} | AI đoán {diem_ai[i]:.1f}")
        if ghi_chu is not None:
            print(f"      {ghi_chu[stt - 1]}")


## BƯỚC 8: Ứng dụng thực tế - Dự đoán điểm, Lọc Top 10 & Học khẩu vị gợi ý Top 5

Trên Jupyter Notebook, thay vì dùng vòng lặp `while True` chạy liên tục dễ làm lặp lại thanh `input()`, Bước 8 được chia thành **2 ô code độc lập chạy nối tiếp nhau (`Bước 8A` và `Bước 8B`)**:
- **Ô Bước 8A:** Nhập thông tin phim 1 lần $\rightarrow$ Dự đoán điểm số & In danh sách **Top 10 phim hợp sở thích**, sau đó tự kết thúc để bạn thoải mái cuộn xem kết quả.
- **Ô Bước 8B:** Nhập số thứ tự phim **Thích / Không thích** từ Top 10 $\rightarrow$ Học khẩu vị và gợi ý thêm **Top 5 phim mới**. Khi muốn thử bộ phim khác, bạn chỉ cần bấm chạy lại ô Bước 8A.



### Giai đoạn 1 (Bước 8A - Phần 1): Dự đoán điểm số cho cấu hình phim người dùng nhập
Vector đầu vào $\mathbf{x}_{\text{mới}} = [\text{năm}, \text{thời lượng}, G_1, \dots, G_{15}]$ được đưa qua bộ `scaler.transform()` (sử dụng đúng $\mu, \sigma$ của tập Train) rồi nhân với trọng số của `LinearRegression` để xuất ra điểm dự kiến.

### Giai đoạn 2 (Bước 8A - Phần 2): Thuật toán Lọc theo nội dung (Content-Based Filtering) tìm Top 10 phim
Với mỗi bộ phim $i$ trong kho dữ liệu, hệ thống tính điểm tương đồng có trọng số:
1. **Độ phủ thể loại (Trọng số $w_{\text{genre}} = 4.0$):**
   $$\text{Coverage}_i = \frac{|C \cap G_i|}{|C|}$$
   Trong đó $C$ là tập thể loại người dùng chọn và $G_i$ là tập thể loại của phim $i$. Nếu phim có đủ mọi thể loại người dùng tìm, $\text{Coverage}_i = 1.0$.
2. **Độ khớp Năm phát hành (Trọng số $w_{\text{year}} = 1.5$, thang đo $\tau = 10$ năm):**
   $$S_{\text{year}, i} = e^{-\frac{|\text{year}_i - \text{year}_{\text{input}}|}{10}}$$
3. **Độ khớp Thời lượng (Trọng số $w_{\text{runtime}} = 1.0$, thang đo $\tau = 30$ phút):**
   $$S_{\text{runtime}, i} = e^{-\frac{|\text{runtime}_i - \text{runtime}_{\text{input}}|}{30}}$$
4. **Công thức Điểm xếp hạng tổng hợp (`diem_xep_hang`):**
   $$\text{Score}_i = \frac{4.0 \cdot \text{Coverage}_i + 1.5 \cdot S_{\text{year}, i} + 1.0 \cdot S_{\text{runtime}, i}}{4.0 + 1.5 + 1.0} + 0.1 \times \frac{\text{Điểm thật}_i}{10}$$
   Thành phần cộng thêm $0.1 \times \frac{\text{Điểm thật}_i}{10}$ (tối đa $0.1$ điểm) đóng vai trò phân định thứ hạng (tie-breaker): giữa các phim có độ khớp ngang nhau, phim nào được cộng đồng đánh giá cao hơn sẽ nổi lên đầu.

In [16]:
# =====================================================================
# BƯỚC 8A: NHẬP SỞ THÍCH -> DỰ ĐOÁN ĐIỂM & TÌM TOP 10 PHIM
# =====================================================================
print("=" * 60)
print("  CHƯƠNG TRÌNH AI DỰ ĐOÁN ĐIỂM & GỢI Ý PHIM ")
print("=" * 60)
print("Thể loại có sẵn:", ", ".join(top_genres))

top_genres_thuong = set()
for g in top_genres:
    top_genres_thuong.add(g.lower())

try:
    the_loai_input = input("\n Nhập Thể loại (cách nhau dấu phẩy, tiếng Anh): ")
    nam_input = input(" Nhập Năm phát hành (VD: 2024): ")
    thoi_luong_input = input(" Nhập Thời lượng (phút): ")

    nam = int(nam_input)
    thoi_luong = float(thoi_luong_input)

    # Tách các thể loại người dùng vừa nhập
    chon = set()
    for tu in the_loai_input.split(','):
        tl = tu.strip().lower()
        if tl != '':
            chon.add(tl)

    # Báo cho người dùng nếu nhập thể loại nằm ngoài 15 thể loại của mô hình
    the_loai_la = chon - top_genres_thuong
    if len(the_loai_la) > 0:
        print(f" (Lưu ý: không có thể loại {', '.join(sorted(the_loai_la))} trong mô hình)")

    # -------------------------------------------------------------
    # PHẦN 1: AI dự đoán điểm cho phim vừa nhập
    # -------------------------------------------------------------
    dong_du_lieu_moi = [nam, thoi_luong]
    for g in top_genres:
        if g.lower() in chon:
            dong_du_lieu_moi.append(1)
        else:
            dong_du_lieu_moi.append(0)

    du_lieu_moi = pd.DataFrame([dong_du_lieu_moi], columns=features)
    du_lieu_moi_scaled = scaler.transform(du_lieu_moi)
    diem_du_doan = float(np.clip(model.predict(du_lieu_moi_scaled)[0], 0, 10))

    print("-" * 60)
    print(f"Phim {the_loai_input or '(không rõ thể loại)'}, năm {nam}, dài {thoi_luong:.0f} phút")
    print(f"ĐIỂM DỰ KIẾN: {diem_du_doan:.2f} / 10")

    # -------------------------------------------------------------
    # PHẦN 2: Tìm TOP 10 phim hợp gu nhất
    # (Trọng số: thể loại = 4.0, năm = 1.5, thời lượng = 1.0)
    # -------------------------------------------------------------
    tong_diem = np.zeros(len(df))
    tong_trong_so = 0.0

    # 1. Tính điểm khớp thể loại
    if len(chon) > 0:
        danh_sach_do_phu = []
        for tl_phim in the_loai_tung_phim:
            so_tl_trung = len(chon & tl_phim)
            ty_le_trung = so_tl_trung / len(chon)
            danh_sach_do_phu.append(ty_le_trung)

        do_phu_the_loai = np.array(danh_sach_do_phu)
        tong_diem += 4.0 * do_phu_the_loai
        tong_trong_so += 4.0

    # 2. Cộng thêm điểm khớp năm phát hành và thời lượng
    tong_diem += 1.5 * do_khop(nam_tat_ca, nam, 10)
    tong_diem += 1.0 * do_khop(tl_tat_ca, thoi_luong, 30)
    tong_trong_so += 2.5

    # 3. Tính điểm xếp hạng chung và lấy ra 10 phim cao nhất
    diem_xep_hang = (tong_diem / tong_trong_so) + (0.1 * diem_that / 10)
    top10 = np.argsort(-diem_xep_hang)[:10]

    print("\n--- TOP 10 PHIM HỢP VỚI SỞ THÍCH CỦA BẠN ---")
    in_danh_sach(top10)

except ValueError:
    print(" Lỗi: Năm và thời lượng phải là chữ số! Vui lòng thử lại.")

  CHƯƠNG TRÌNH AI DỰ ĐOÁN ĐIỂM & GỢI Ý PHIM 
Thể loại có sẵn: Drama, Comedy, Thriller, Action, Romance, Horror, Crime, Adventure, Animation, Family, Science Fiction, Fantasy, Mystery, Documentary, TV Movie
------------------------------------------------------------
Phim Drama, Comedy, Action, Animation, năm 2020, dài 120 phút
ĐIỂM DỰ KIẾN: 7.32 / 10

--- TOP 10 PHIM HỢP VỚI SỞ THÍCH CỦA BẠN ---
 1. Masameer: The Movie (2020, 112 phút)
      Thể loại: Animation, Comedy, Action, Drama
      Điểm thật 6.1 | AI đoán 7.3
 2. Steven Universe: The Movie (2019, 82 phút)
      Thể loại: TV Movie, Animation, Science Fiction, Action, Adventure, Comedy, Drama, Fantasy, Music
      Điểm thật 8.3 | AI đoán 6.9
 3. Pretty Guardian Sailor Moon Eternal The Movie Part (2021, 80 phút)
      Thể loại: Animation, Action, Adventure, Comedy, Drama, Fantasy, Romance
      Điểm thật 8.1 | AI đoán 7.0
 4. Fate/stay night: Heaven's Feel III. Spring Song (2020, 120 phút)
      Thể loại: Animation, Romance, Fanta


### Giai đoạn 3 (Bước 8B): Học khẩu vị tương tác bằng Hồi quy Ridge (`Ridge Regression`) & Giải thích AI (XAI)
Khi người dùng chọn các phim **THÍCH** (gán nhãn $+1.0$) và **KHÔNG THÍCH** (gán nhãn $-1.0$) từ Top 10:
1. **Tại sao dùng `Ridge(alpha=10.0)` để học vector khẩu vị $w$?**
   - Số lượng phim người dùng chọn rất ít ($k \le 10$ mẫu), trong khi không gian đặc trưng có tới $17$ chiều ($k < d$). Nếu dùng Hồi quy tuyến tính thường (OLS), hệ phương trình sẽ suy biến (Underdetermined) và sinh ra trọng số cực đoan (Overfitting).
   - **Hồi quy Ridge** giải bài toán tối ưu có phạt chuẩn L2 (Tikhonov Regularization):
     $$\min_{w} \|X_{\text{chọn}} w - y_{\text{nhãn}}\|_2^2 + \alpha \|w\|_2^2 \quad (\alpha = 10.0)$$
     Nghiệm giải tích $w = (X_{\text{chọn}}^T X_{\text{chọn}} + \alpha I)^{-1} X_{\text{chọn}}^T y_{\text{nhãn}}$ luôn tồn tại duy nhất, ổn định và phản ánh mượt mà xu hướng thích/ghét của người dùng trên 17 đặc trưng.
   - **Trường hợp suy biến 1 nhãn** (người dùng chỉ chọn toàn phim thích hoặc chỉ chọn toàn phim ghét): Vector khẩu vị $w$ được tính trực tiếp bằng vector trung bình mẫu (Prototype Vector / Rocchio Centroid):
     $$w = \pm \frac{1}{|S|} \sum_{i \in S} \mathbf{x}_i$$
2. **Chấm điểm gợi ý Top 5 phim mới (`diem_goi_y`):**
   - Tính độ khớp khẩu vị cho toàn bộ kho phim qua tích vô hướng ma trận: `diem_khau_vi = X_all_scaled @ w`.
   - Kết hợp điểm khẩu vị (chủ đạo) và chất lượng phim thực tế (trọng số $0.3$) sau khi chuẩn hóa Z-Score:
     $$\text{Điểm gợi ý}_i = Z(\text{Điểm khẩu vị}_i) + 0.3 \cdot Z(\text{Điểm thật}_i)$$
   - Loại trừ 10 phim đã hiển thị bằng cách gán `diem_goi_y[top10] = -1e9`.
3. **Giải thích lý do gợi ý (Explainable AI - Feature Attribution):**
   - Với mỗi phim $i$ trong Top 5, ta chỉ xét các đặc trưng hiện diện dương ở bộ phim đó ($\max(X_{i,j}, 0)$) và nhân với trọng số khẩu vị $w_j$:
     $$\text{Contribution}_{i,j} = w_j \times \max(X_{i,j}, 0)$$
   - Lấy tối đa 3 đặc trưng có $\text{Contribution}_{i,j} > 0$ lớn nhất để ghép thành câu giải thích tự nhiên: *"Vì bạn có xu hướng thích: thể loại ..., năm phát hành ..., thời lượng ... phút"*.

In [17]:
# =====================================================================
# BƯỚC 8B: HỌC KHẨU VỊ TỪ TOP 10 & GỢI Ý THÊM 5 PHIM MỚI
# =====================================================================
print("Hãy cho AI biết khẩu vị của bạn trong 10 phim trên (Enter = bỏ qua):")
thich = doc_so_phim(input(" Bạn THÍCH phim số nào? (VD: 1 3 5): "), 10)
nhap_khong_thich = doc_so_phim(input(" Bạn KHÔNG THÍCH phim số nào? (VD: 2 7): "), 10)

# Lọc bỏ nếu người dùng lỡ nhập cùng 1 số ở cả 2 dòng
khong_thich = []
for vi_tri in nhap_khong_thich:
    if vi_tri not in thich:
        khong_thich.append(vi_tri)

if len(thich) == 0 and len(khong_thich) == 0:
    print(" (Bạn chưa chọn phim nào nên AI chưa gợi ý được thêm.)")
else:
    # Gán nhãn: phim thích = +1.0, phim không thích = -1.0
    vi_tri_da_chon = thich + khong_thich
    nhan = np.array([1.0] * len(thich) + [-1.0] * len(khong_thich))
    X_da_chon = X_all_scaled[top10[vi_tri_da_chon]]

    # Tìm vector khẩu vị (w) của người dùng
    if len(thich) > 0 and len(khong_thich) > 0:
        mo_hinh_khau_vi = Ridge(alpha=10.0)
        mo_hinh_khau_vi.fit(X_da_chon, nhan)
        w = mo_hinh_khau_vi.coef_
    else:
        he_so_dau = 1.0 if len(thich) > 0 else -1.0
        w = X_da_chon.mean(axis=0) * he_so_dau

    # Chấm điểm khẩu vị cho toàn bộ kho phim và lấy Top 5
    diem_khau_vi = X_all_scaled @ w
    diem_goi_y = chuan_hoa_z(diem_khau_vi) + 0.3 * chuan_hoa_z(diem_that)
    diem_goi_y[top10] = -1e9  # Tránh gợi ý trùng lại 10 phim vừa hiện ở trên
    top5 = np.argsort(-diem_goi_y)[:5]

    # Tạo câu giải thích lý do gợi ý cho từng phim trong Top 5
    ly_do = []
    for i in top5:
        dac_trung_duong = np.maximum(X_all_scaled[i], 0)
        dong_gop = w * dac_trung_duong
        ba_vi_tri_cao_nhat = np.argsort(-dong_gop)[:3]

        cac_yeu_to = []
        for j in ba_vi_tri_cao_nhat:
            if dong_gop[j] > 0:
                cac_yeu_to.append(ten_dep(features[j], i))

        if len(cac_yeu_to) > 0:
            chuoi_ly_do = ", ".join(cac_yeu_to)
        else:
            chuoi_ly_do = "(không rõ)"

        ly_do.append("Vì bạn có xu hướng thích: " + chuoi_ly_do)

    print("\n--- AI GỢI Ý THÊM 5 PHIM DÀNH CHO BẠN ---")
    in_danh_sach(top5, ly_do)
    print("-" * 60)

Hãy cho AI biết khẩu vị của bạn trong 10 phim trên (Enter = bỏ qua):

--- AI GỢI Ý THÊM 5 PHIM DÀNH CHO BẠN ---
 1. Regular Show: The Movie (2015, 70 phút)
      Thể loại: Animation, Comedy, Science Fiction, TV Movie, Adventure, Action
      Điểm thật 7.9 | AI đoán 6.5
      Vì bạn có xu hướng thích: thể loại Adventure, thể loại TV Movie, thể loại Comedy
 2. Family Guy Presents: Stewie Kills Lois and Lois Ki (2007, 45 phút)
      Thể loại: Animation, Comedy, Action, Adventure, Crime, Thriller, Music, TV Movie
      Điểm thật 8.1 | AI đoán 6.5
      Vì bạn có xu hướng thích: thể loại Adventure, thể loại TV Movie, thể loại Comedy
 3. Barbie and the Sensations: Rockin' Back to Earth (1987, 25 phút)
      Thể loại: TV Movie, Animation, Music, Adventure, Science Fiction
      Điểm thật 7.6 | AI đoán 6.4
      Vì bạn có xu hướng thích: thể loại Adventure, thể loại TV Movie, thể loại Science Fiction
 4. Billy & Mandy's Big Boogey Adventure (2007, 80 phút)
      Thể loại: Adventure, Comedy, An